# Pipeline 2 : Traduire les molecules en nombres

La question du chercheur : un ordinateur ne comprend pas une structure chimique dessinee, comment lui donner de quoi raisonner sur une molecule ?

Une molecule est decrite dans nos donnees par une chaine de caracteres appelee SMILES, par exemple CCO pour l'ethanol. C'est lisible pour un chimiste mais inutilisable tel quel par un modele de machine learning. Il faut transformer chaque molecule en une liste de nombres, ce qu'on appelle des descripteurs.

On va calculer deux familles de descripteurs completement differentes. D'abord les descripteurs physico-chimiques, qui resument les grandes proprietes d'une molecule comme sa taille ou sa solubilite. Ensuite les empreintes moleculaires, ou fingerprints, qui encodent la presence ou l'absence de petits motifs structurels. Ces deux representations se completent, et on verra plus tard laquelle marche le mieux pour predire l'activite.

In [ ]:
# Sur Colab, decommenter pour installer
# !pip install rdkit -q

import warnings
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Draw, AllChem
from rdkit import DataStructs

C_BLEU = "#1f6f8b"
C_ORANGE = "#e0771a"
C_VERT = "#2e8b57"
C_ROUGE = "#9b2226"
C_GRIS = "#8d99ae"

plt.rcParams.update({
    "figure.figsize": (13, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 13
})

RANDOM_STATE = 42

# 1. Chargement des donnees preparees

In [ ]:
df = pd.read_csv('egfr_donnees_completes.csv')
print(f"Molecules chargees : {len(df)}")
df.head()

# 2. Convertir les SMILES en objets molecules RDKit

RDKit transforme chaque chaine SMILES en un objet molecule sur lequel on peut faire des calculs. Certaines chaines peuvent etre invalides ou mal formees, on les detecte et on les retire pour ne pas propager d'erreurs.

In [ ]:
df['mol'] = df['canonical_smiles'].apply(Chem.MolFromSmiles)
n_invalides = df['mol'].isnull().sum()
df = df[df['mol'].notnull()].reset_index(drop=True)

print(f"Molecules invalides retirees : {n_invalides}")
print(f"Molecules valides conservees : {len(df)}")

# 3. Les descripteurs de Lipinski et physico-chimiques

On calcule les grandeurs les plus parlantes pour un chimiste. Le poids moleculaire mesure la taille de la molecule. Le LogP mesure son caractere gras ou hydrophobe, ce qui gouverne sa capacite a traverser les membranes cellulaires. Le nombre de donneurs et d'accepteurs de liaisons hydrogene decrit comment elle interagit avec l'eau et avec sa cible. La surface polaire, ou TPSA, est liee a son absorption par l'organisme. Le nombre de liaisons rotatives mesure sa flexibilite.

In [ ]:
def calculer_descripteurs(mol):
    return pd.Series({
        'poids_moleculaire': Descriptors.MolWt(mol),
        'logP': Descriptors.MolLogP(mol),
        'donneurs_H': Lipinski.NumHDonors(mol),
        'accepteurs_H': Lipinski.NumHAcceptors(mol),
        'tpsa': Descriptors.TPSA(mol),
        'liaisons_rotatives': Descriptors.NumRotatableBonds(mol),
        'anneaux_aromatiques': Lipinski.NumAromaticRings(mol)
    })

descripteurs = df['mol'].apply(calculer_descripteurs)
df = pd.concat([df, descripteurs], axis=1)
gc.collect()

print("Descripteurs physico-chimiques calcules :")
print(df[['poids_moleculaire', 'logP', 'donneurs_H', 'accepteurs_H',
          'tpsa', 'liaisons_rotatives', 'anneaux_aromatiques']].describe().round(2))

# 4. La regle des cinq de Lipinski

Christopher Lipinski a formule dans les annees 1990 une regle empirique qui predit si une molecule a des chances d'etre un bon medicament administrable par voie orale. Une molecule respecte la regle si son poids est inferieur a 500, son LogP inferieur a 5, ses donneurs de liaisons hydrogene au nombre de 5 au plus, et ses accepteurs au nombre de 10 au plus. Une molecule qui viole plusieurs de ces criteres sera difficile a transformer en comprime.

In [ ]:
df['viol_lipinski'] = (
    (df['poids_moleculaire'] > 500).astype(int) +
    (df['logP'] > 5).astype(int) +
    (df['donneurs_H'] > 5).astype(int) +
    (df['accepteurs_H'] > 10).astype(int)
)
df['respecte_lipinski'] = df['viol_lipinski'] == 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['viol_lipinski'].value_counts().sort_index()
bars = axes[0].bar(counts.index.astype(str), counts.values, color=C_BLEU, edgecolor='white')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                 f"{bar.get_height()}", ha='center', fontsize=10)
axes[0].set_title("Nombre de violations de la regle de Lipinski")
axes[0].set_xlabel("Nombre de criteres violes")
axes[0].set_ylabel("Nombre de molecules")

taux = df['respecte_lipinski'].mean() * 100
axes[1].pie([taux, 100 - taux], labels=['Respecte', 'Viole'],
            colors=[C_VERT, C_GRIS], autopct='%1.0f%%', startangle=90,
            wedgeprops={'edgecolor': 'white'})
axes[1].set_title("Part des molecules conformes a Lipinski")

plt.tight_layout()
plt.show()

print(f"Environ {taux:.0f}% des molecules testees sur l'EGFR respectent parfaitement la regle des cinq. C'est logique, ce sont des molecules concues par des chimistes medicinaux qui connaissent ces criteres. Les quelques molecules qui violent plusieurs regles sont souvent de grosses structures exploratoires, interessantes scientifiquement mais peu susceptibles de devenir des medicaments oraux en l'etat. Pour un chercheur, cette information oriente d'emblee le tri.")

# 5. Les empreintes moleculaires (Morgan fingerprints)

Les descripteurs physico-chimiques resument la molecule en quelques nombres, mais ils perdent la structure fine. Deux molecules tres differentes peuvent avoir le meme poids et le meme LogP. Les empreintes de Morgan resolvent ce probleme d'une autre maniere : elles parcourent chaque atome et son voisinage, et enregistrent la presence de chaque petit motif structurel dans un long vecteur de 0 et de 1. Deux molecules qui partagent beaucoup de motifs auront des empreintes proches. On utilise un rayon de 2 et 2048 bits, un standard de la cheminformatique.

In [ ]:
def calculer_fingerprint(mol, rayon=2, n_bits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, rayon, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fingerprints = np.array([calculer_fingerprint(m) for m in df['mol']])
gc.collect()

print(f"Matrice d'empreintes : {fingerprints.shape[0]} molecules x {fingerprints.shape[1]} bits")
densite = fingerprints.mean() * 100
print(f"En moyenne, {densite:.1f}% des bits sont actives par molecule, le reste est a zero.")
print("Cette matrice creuse est ce que les modeles de classification utiliseront comme entree structurelle.")

# 6. Visualiser quelques molecules

Un des avantages de RDKit est qu'il sait redessiner les structures a partir des SMILES. On affiche les molecules les plus actives de notre jeu de donnees, celles qui inhibent le plus puissamment l'EGFR.

In [ ]:
top_actives = df.nlargest(6, 'pIC50')
legendes = [f"pIC50 = {p:.2f}" for p in top_actives['pIC50']]

img = Draw.MolsToGridImage(
    top_actives['mol'].tolist(),
    molsPerRow=3,
    subImgSize=(280, 220),
    legends=legendes
)
img

In [ ]:
print("Voici les six molecules les plus puissantes contre l'EGFR dans notre base. On remarque des points communs structurels a l'oeil nu, notamment des systemes d'anneaux aromatiques accoles typiques des inhibiteurs de kinases. Ces motifs recurrents chez les molecules actives sont precisement ce que les modeles vont apprendre a reconnaitre, et ce que le pipeline d'analyse des scaffolds formalisera plus tard.")

# 7. Sauvegarde des descripteurs

On sauvegarde le tableau des descripteurs physico-chimiques et la matrice des empreintes. Les notebooks de modelisation rechargeront directement ces representations numeriques.

In [ ]:
colonnes_desc = ['canonical_smiles', 'molecule_chembl_id', 'pIC50',
                 'poids_moleculaire', 'logP', 'donneurs_H', 'accepteurs_H',
                 'tpsa', 'liaisons_rotatives', 'anneaux_aromatiques',
                 'viol_lipinski', 'respecte_lipinski']
df[colonnes_desc].to_csv('egfr_descripteurs.csv', index=False)
np.save('egfr_fingerprints.npy', fingerprints.astype(np.int8))

print("Fichiers sauvegardes :")
print("  egfr_descripteurs.csv : descripteurs physico-chimiques par molecule")
print("  egfr_fingerprints.npy : matrice des empreintes de Morgan")

# Conclusion

On dispose maintenant de deux langages pour decrire nos molecules. Les descripteurs physico-chimiques sont peu nombreux et interpretables, parfaits pour comprendre ce qui distingue une molecule active. Les empreintes de Morgan sont massives et opaques mais capturent la structure fine, souvent plus performantes en prediction pure.

La question de savoir laquelle des deux representations, ou leur combinaison, donne les meilleurs modeles n'est pas tranchee a ce stade, et c'est justement ce qu'on testera dans le notebook de classification. La limite a garder en tete est que les empreintes, malgre leur richesse, restent aveugles a la geometrie tridimensionnelle des molecules, qui joue pourtant un role dans l'activite reelle.